# Python Closures and Scopes

This notebook provides a comprehensive exploration of Python closures and scopes, which are fundamental concepts for understanding how Python manages variable namespaces and creates specialized functions.

## 1. Understanding Variable Scopes in Python

Python uses a scoping rule called the LEGB rule to determine how to look up names in the code:

- **Local (L)**: The local scope refers to variables defined within the current function
- **Enclosing (E)**: The enclosing scope refers to variables defined in an outer function that contains a nested function 
- **Global (G)**: The global scope refers to variables defined at the top level of a module or declared global in a function
- **Built-in (B)**: The built-in scope refers to names preassigned in Python (like `print`, `len`, etc.)

Let's see how variable lookup works in different scopes:

In [ ]:
# Global scope example
x = 10  # Global variable

def test_function():
    y = 5  # Local variable
    print(f"Inside function - x: {x}, y: {y}")
    
print(f"Global scope - x: {x}")
test_function()

# This will raise an error because y is a local variable inside test_function
try:
    print(f"Global scope - y: {y}")
except NameError as e:
    print(f"Error: {e}")

In the example above:
- `x` is defined in the global scope and accessible everywhere
- `y` is defined in the local scope of `test_function` and only accessible within that function

The LEGB rule dictates the order Python uses to search for variables. Let's see a more comprehensive example:

In [ ]:
# Built-in scope
print(len("Hello"))  # Using built-in len() function

# Global scope
x = "global x"

def outer():
    # Enclosing scope
    x = "outer x"
    
    def inner():
        # Local scope
        # x = "inner x"  # Uncomment to see how local scope works
        print("inner:", x)
    
    inner()
    print("outer:", x)

outer()
print("global:", x)

## 2. Nested Functions

Python allows functions to be defined inside other functions. These are called nested functions or inner functions. The inner function can access variables from its enclosing function's scope.

In [ ]:
def outer_function(message):
    # This is the outer function
    outer_variable = "I'm from the outer function"
    
    def inner_function():
        # This is the inner function
        print(message)  # Can access the parameter of outer_function
        print(outer_variable)  # Can access variables defined in outer_function
    
    # Call the inner function
    inner_function()

# Test the nested function
outer_function("Hello from the nested function!")

Notice that the inner function can access:
1. Its own local variables
2. Variables defined in the enclosing function
3. Global variables
4. Built-in names

However, the inner function cannot modify the enclosing function's variables unless we use the `nonlocal` keyword (which we'll see later).

In [ ]:
def demonstrate_scope_access():
    outer_var = 100
    
    def inner_function():
        # Can access but not modify outer_var (without nonlocal)
        print(f"Inner function can access outer_var: {outer_var}")
        
        # Let's try to modify it
        # outer_var = 200  # Uncommenting this will cause an error
        # The above line creates a new local variable instead of modifying the outer one
    
    inner_function()
    print(f"Value of outer_var remains: {outer_var}")

demonstrate_scope_access()

## 3. Closures Fundamentals

A closure is a function object that remembers values in the enclosing scope even if they are not present in memory. In simpler terms, a closure is a function that captures and "closes over" the variables from its surrounding lexical scope.

For a closure to exist, three conditions must be met:
1. We must have a nested function (a function defined inside another function)
2. The nested function must refer to a value defined in the enclosing function
3. The enclosing function must return the nested function

Closures allow us to create specialized functions with "built-in" data.

In [ ]:
def make_multiplier(x):
    # This outer function takes parameter x
    
    def multiply(y):
        # This inner function uses x from the outer function
        return x * y
    
    # Return the inner function, which forms a closure
    return multiply

# Create two different multiplier functions
double = make_multiplier(2)
triple = make_multiplier(3)

# Use the closure functions
print(double(5))  # 2 * 5 = 10
print(triple(5))  # 3 * 5 = 15

# The closures maintain their own copies of the outer function's variable
print(double(10))  # 2 * 10 = 20
print(triple(10))  # 3 * 10 = 30

What happens when `make_multiplier(2)` is called?
1. A new function object `multiply` is created
2. A reference to variable `x` with value `2` is stored in the function object
3. This function object is returned and assigned to `double`

Even though `make_multiplier` has finished executing, the `double` function still has access to the `x` value through the closure.

## 4. Creating and Using Closures

Let's explore more practical examples of creating and using closures. We'll see how closures can maintain state between function calls.

In [ ]:
def counter():
    count = 0
    
    def increment():
        nonlocal count  # This is needed to modify the enclosed variable
        count += 1
        return count
    
    return increment

# Create a counter
my_counter = counter()

# Call it multiple times to see how it maintains state
print(my_counter())  # 1
print(my_counter())  # 2
print(my_counter())  # 3

# Create a second counter which will have its own state
counter2 = counter()
print(counter2())  # 1

# First counter continues from where it left off
print(my_counter())  # 4

Closures are useful when:
1. You want to avoid using global variables
2. You want to create functions with "memory"
3. You want to implement data hiding or encapsulation
4. You want to create specialized function templates

Here's another example that creates a function to track running averages:

In [ ]:
def running_average():
    total = 0
    count = 0
    
    def average(new_value):
        nonlocal total, count
        total += new_value
        count += 1
        return total / count
    
    return average

# Create an average function
avg = running_average()

# Add values and get running average
print(avg(10))  # 10.0
print(avg(20))  # 15.0
print(avg(30))  # 20.0
print(avg(40))  # 25.0

## 5. Nonlocal and Global Keywords

Python provides two special keywords to work with variables in different scopes:

1. `nonlocal`: Used to indicate that a variable refers to a name in the nearest enclosing scope (excluding globals)
2. `global`: Used to indicate that a variable refers to a name in the global scope

Let's see how they work:

In [ ]:
# Example with nonlocal
def outer():
    x = "outer"
    
    def inner_without_nonlocal():
        # This creates a new local variable x
        x = "inner"
        print("inner_without_nonlocal:", x)
    
    def inner_with_nonlocal():
        nonlocal x  # This refers to the x in the outer function
        x = "modified by inner"
        print("inner_with_nonlocal:", x)
    
    print("Before any calls:", x)
    inner_without_nonlocal()
    print("After inner_without_nonlocal:", x)
    inner_with_nonlocal()
    print("After inner_with_nonlocal:", x)

outer()

In [ ]:
# Example with global
global_var = "I'm global"

def modify_global():
    # Using global variable without modification
    print("Inside function (before):", global_var)
    
    # This would create a local variable instead of modifying the global one
    # global_var = "Modified"  # Uncommenting this will cause an error
    
    # To modify the global variable:
    global global_var
    global_var = "Modified global"
    print("Inside function (after):", global_var)

print("Before function call:", global_var)
modify_global()
print("After function call:", global_var)

Key points about `nonlocal` and `global`:

- Use `nonlocal` when you need to modify a variable in an enclosing (but non-global) scope
- Use `global` when you need to modify a variable in the global scope
- Without these keywords, assigning to a variable inside a function creates a new local variable
- Both keywords must be used before any assignments to the variable in that function

## 6. Common Use Cases for Closures

Closures are extremely useful in many programming scenarios. Let's explore some common use cases:

In [ ]:
# 1. Data hiding / private variables
def create_account(initial_balance):
    balance = initial_balance
    
    def deposit(amount):
        nonlocal balance
        balance += amount
        return balance
    
    def withdraw(amount):
        nonlocal balance
        if amount > balance:
            return "Insufficient funds"
        balance -= amount
        return balance
    
    def check_balance():
        return balance
    
    # Return a dictionary of functions (like an interface)
    return {
        "deposit": deposit,
        "withdraw": withdraw,
        "check_balance": check_balance
    }

# Create a bank account with closure
account = create_account(1000)
print(f"Initial balance: {account['check_balance']()}")
print(f"After deposit: {account['deposit'](500)}")
print(f"After withdrawal: {account['withdraw'](200)}")
print(f"Attempt to withdraw too much: {account['withdraw'](2000)}")

# The balance variable is not directly accessible (it's "private")
# print(account.balance)  # This would raise an error

In [ ]:
# 2. Creating function decorators
def log_calls(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with args: {args}, kwargs: {kwargs}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
        return result
    return wrapper

@log_calls
def add(a, b):
    return a + b

# Using the decorated function
add(3, 5)

In [ ]:
# 3. Memoization (caching function results)
def memoize(func):
    cache = {}
    
    def wrapper(*args):
        if args in cache:
            print(f"Cache hit for {args}")
            return cache[args]
        else:
            print(f"Cache miss for {args}, calculating...")
            result = func(*args)
            cache[args] = result
            return result
    
    return wrapper

@memoize
def fibonacci(n):
    if n <= 1:
        return n
    else:
        return fibonacci(n-1) + fibonacci(n-2)

# This will calculate each value only once
print(fibonacci(10))
print(fibonacci(10))  # Second call will be much faster due to caching

## 7. Function Factories

Function factories use closures to create customized function instances. They are functions that create and return other functions with specific behaviors based on the parameters provided.

In [ ]:
# Function factory to create power functions
def make_power_function(exponent):
    def power_function(base):
        return base ** exponent
    
    return power_function

# Create specific power functions
square = make_power_function(2)
cube = make_power_function(3)
square_root = make_power_function(0.5)

# Test the functions
print(f"Square of 4: {square(4)}")
print(f"Cube of 3: {cube(3)}")
print(f"Square root of 16: {square_root(16)}")

In [ ]:
# Function factory for filtering with different criteria
def create_filter(keyword):
    def filter_function(items):
        return [item for item in items if keyword.lower() in item.lower()]
    return filter_function

# Create specialized filter functions
filter_apple = create_filter("apple")
filter_fruit = create_filter("fruit")

# Test the filters
fruits = ["Apple", "Banana", "Apple pie", "Orange", "Mixed fruit salad"]
print(f"Items containing 'apple': {filter_apple(fruits)}")
print(f"Items containing 'fruit': {filter_fruit(fruits)}")

In [ ]:
# Function factory for creating formatters
def create_formatter(template):
    def formatter(**kwargs):
        return template.format(**kwargs)
    return formatter

# Create specific formatters
greeting = create_formatter("Hello, {name}! Welcome to {place}.")
email = create_formatter("Dear {title} {surname},\n\nThank you for your {topic} submission.\n\nBest regards,\n{sender}")

# Use the formatters
print(greeting(name="Alice", place="Wonderland"))
print("\n" + email(title="Dr.", surname="Smith", topic="research", sender="Conference Committee"))

## 8. Closures vs. Classes

Closures and classes can often be used to solve the same problems, especially when it comes to maintaining state. Let's compare the two approaches:

In [ ]:
# Implement a counter using both closures and classes

# 1. Using closure
def create_counter_closure():
    count = 0
    
    def increment():
        nonlocal count
        count += 1
        return count
        
    def get_count():
        return count
    
    def reset():
        nonlocal count
        count = 0
        return count
    
    return {
        "increment": increment,
        "get_count": get_count,
        "reset": reset
    }

# 2. Using class
class CounterClass:
    def __init__(self):
        self.count = 0
    
    def increment(self):
        self.count += 1
        return self.count
    
    def get_count(self):
        return self.count
    
    def reset(self):
        self.count = 0
        return self.count

# Test closure-based counter
counter_closure = create_counter_closure()
print("Closure counter:")
print(counter_closure["increment"]())
print(counter_closure["increment"]())
print(f"Current count: {counter_closure['get_count']()}")
print(f"After reset: {counter_closure['reset']()}")

# Test class-based counter
counter_class = CounterClass()
print("\nClass counter:")
print(counter_class.increment())
print(counter_class.increment())
print(f"Current count: {counter_class.get_count()}")
print(f"After reset: {counter_class.reset()}")

### Comparison between Closures and Classes:

**Advantages of Closures:**
- Simpler syntax for simple cases
- Can be more memory-efficient for many instances
- Good for creating one-off specialized functions
- More functional programming style

**Advantages of Classes:**
- Better for complex behavior with many methods
- More explicit about attributes and methods
- Standard OOP pattern that many developers recognize
- Built-in support for inheritance and polymorphism
- Better for serialization and introspection

**When to use which:**
- Use closures for simple function templates or when you need lightweight function objects
- Use classes for complex behaviors, when you need inheritance, or when you want more explicit state management

## 9. Debugging Closures

Debugging closures can be challenging because they maintain state that's not immediately visible. Python provides some tools to help inspect closures:

In [ ]:
def create_power_function(exponent):
    def power(base):
        return base ** exponent
    return power

# Create a closure
square = create_power_function(2)

# Inspect the closure
print(f"Function name: {square.__name__}")
print(f"Function qualname: {square.__qualname__}")
print(f"Function docstring: {square.__doc__}")
print(f"Function closure: {square.__closure__}")

# Examine the values inside the closure
if square.__closure__:
    for i, cell in enumerate(square.__closure__):
        print(f"Closure cell {i}: {cell.cell_contents}")

### Common Debugging Issues with Closures:

1. **Late Binding Closures**: This is probably the most common issue with closures in Python.

In [ ]:
# Example of a late binding closure problem
def create_multipliers_wrong():
    multipliers = []
    for i in range(1, 6):
        # This doesn't work as expected because 'i' is evaluated when the function is called,
        # not when it's defined
        multipliers.append(lambda x: x * i)
    return multipliers

# The correct way to handle this
def create_multipliers_correct():
    multipliers = []
    for i in range(1, 6):
        # Create a default parameter with the current value of i
        multipliers.append(lambda x, i=i: x * i)
    return multipliers

wrong_multipliers = create_multipliers_wrong()
correct_multipliers = create_multipliers_correct()

# Test with a value
test_value = 10
print("Using wrong multipliers:")
for m in wrong_multipliers:
    print(m(test_value))  # All will be 10 * 5 = 50

print("\nUsing correct multipliers:")
for m in correct_multipliers:
    print(m(test_value))  # Will correctly print 10, 20, 30, 40, 50

2. **Modifying Free Variables**: Understanding when to use `nonlocal` to modify enclosed variables.

In [ ]:
def create_incrementers():
    # Wrong version (creates new local variables)
    def wrong_incrementer():
        x = 0
        def increment():
            x += 1  # This will fail
            return x
        return increment
    
    # Correct version using nonlocal
    def correct_incrementer():
        x = 0
        def increment():
            nonlocal x
            x += 1
            return x
        return increment
    
    # Return both for comparison
    return {
        "wrong": wrong_incrementer,
        "correct": correct_incrementer
    }

incrementers = create_incrementers()

try:
    wrong_inc = incrementers["wrong"]()
    print(wrong_inc())
except UnboundLocalError as e:
    print(f"Error with wrong incrementer: {e}")

# The correct one works fine
correct_inc = incrementers["correct"]()
print(f"First call: {correct_inc()}")
print(f"Second call: {correct_inc()}")

## 10. Best Practices

When working with closures, follow these best practices for more readable and maintainable code:

### 1. Keep closures small and focused
Closures should have a clear purpose. If a closure becomes too complex, consider using a class instead.

### 2. Document closures well
Since closures capture and use state in a way that's not always obvious, good documentation is essential.

### 3. Be careful with mutable enclosed variables
Changes to mutable objects (like lists or dictionaries) in a closure will affect all references to that closure.

### 4. Watch out for late binding
Use default parameters to capture current values in loop-created closures.

### 5. Use meaningful names
Choose function and variable names that clearly indicate the purpose of the closure.

### 6. Know when to use `nonlocal`
Use `nonlocal` when you need to modify enclosed variables, but be mindful that excessive use can make code harder to understand.

### 7. Consider alternatives
For complex behaviors, consider whether a class or a different pattern might be more appropriate.

In [ ]:
# Example of good closure practice
def create_logger(log_level):
    """
    Create a logging function for the specified log level.
    
    Args:
        log_level (str): The logging level (e.g., 'INFO', 'ERROR')
        
    Returns:
        function: A closure that logs messages with the specified level
    """
    # Capture the current time of creation for reference
    import time
    creation_time = time.strftime("%Y-%m-%d %H:%M:%S")
    
    def log(message):
        """
        Log a message with the specified log level.
        
        Args:
            message (str): The message to log
        """
        import time
        current_time = time.strftime("%H:%M:%S")
        print(f"[{current_time}] {log_level}: {message}")
        
    # Add metadata to the function for better debugging
    log.__name__ = f"{log_level.lower()}_logger"
    log.level = log_level
    log.created_at = creation_time
    
    return log

# Create specific loggers
info_logger = create_logger("INFO")
error_logger = create_logger("ERROR")

# Use the loggers
info_logger("Application started")
error_logger("Failed to connect to database")

# We can inspect our functions
print(f"\nLogger name: {info_logger.__name__}")
print(f"Logger level: {info_logger.level}")
print(f"Logger created at: {info_logger.created_at}")

## Summary

Closures are a powerful feature in Python that allow functions to maintain state and create specialized behavior. They are created when an inner function references variables from an enclosing scope and the inner function is returned.

Key takeaways:
- Python follows the LEGB (Local, Enclosing, Global, Built-in) rule for variable scope resolution
- Closures "close over" variables from their defining environment
- Use `nonlocal` to modify variables in an enclosing scope
- Use `global` to modify global variables
- Closures are useful for data hiding, function factories, decorators, and maintaining state
- Choose between closures and classes based on the complexity of your use case
- Be aware of common pitfalls like late binding and mutable enclosed variables

Mastering closures and scopes in Python will significantly improve your ability to write efficient, elegant, and powerful code.